# Experiment 4 — Late Fusion (RGB MiT-B1 + Tissue CNN) — Colab Run

RGB and the 5 tissue-probability channels are encoded by **separate** networks (ImageNet MiT-B1 for RGB, a small 3-layer CNN for tissue), each pooled to a feature vector, combined only after that — contrast with exp-3 (early fusion: concatenated at the input, one shared conv stack).

Uses the same `datasets/wound-staging-masks-exp-3/` data as exp-3 (same leakage-safe tissue predictions, same verified PIID 3-fold split) — no new zip to upload if exp-3's is already on Drive.

**Run order:** `concat` fusion first (cell 6). Only if that beats the RGB baseline / exp-3 does `gated` fusion (cell 7) become worth running — per the brief, don't skip straight to the more complex variant.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Same `wound-staging-masks-exp-3.zip` as exp-3 — if you already uploaded it to `drive/MyDrive/thesis/` for that run, nothing new to upload here.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp4_late_fusion/scripts")
print(os.getcwd())

## 4b. Shape/gradient sanity checks

Run once before trusting any training result -- checks both fusion modes: output shapes, finite loss, gradients reach the tissue CNN, gradients reach the RGB backbone by default, and freezing the RGB backbone actually stops its gradients.

In [ ]:
!python test_late_fusion_model.py

## 5. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp4_late_fusion_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Train — concat fusion (run this first)

No `--batch` override: uses `configs/base.yaml`'s `batch_size: 16`, matching the baseline exactly. `freeze_epochs=0` by default (single-phase training, same protocol as the baseline/exp-3) -- append `--freeze-epochs N` here if you want to try the staged freeze-then-fine-tune recipe instead.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--fusion", "concat", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("concat fusion training finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 7. Train — gated fusion (only if concat beat the baseline/exp-3)

Per the brief: don't run this as a default, run it as a follow-up once concat's results justify the extra complexity.

In [ ]:
cmd = ["python", "-u", "train.py", "--fusion", "gated", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("gated fusion training finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 8. Check results

3 rows per fusion mode (one per fold) once training completes.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
display(df.groupby("config_id")[["accuracy", "macro_f1", "qwk"]].agg(["mean", "std"]))